# Tucano 2 como PROFESSOR offline do Apolo (esboço funcional)

> ## ⚠️ AVISO SOBRE LICENÇA DA SAÍDA PARA TREINO (verificado em 23/10/2026, não é aconselhamento jurídico)
> - Os pesos do Tucano 2 (Base, Instruct e Think; 0,5B, 1,5B, 3,7B) são **Apache-2.0** ([cartão](https://huggingface.co/Polygl0t/Tucano2-qwen-1.5B-Instruct)). A licença Apache-2.0 **não tem cláusula** proibindo usar as saídas do modelo para treinar outro modelo, nem exige atribuição sobre o texto gerado. Diferente de termos de API (OpenAI, Anthropic etc.), aqui não há restrição contratual do tipo "não use a saída para treinar concorrentes".
> - O cartão diz que o uso previsto é pesquisa e desenvolvimento de modelos de português e que você pode ajustar/adaptar se seguir a Apache-2.0.
> - Cadeia de proveniência: o Tucano2-qwen foi pré-treinado a partir do Qwen3 (Apache-2.0) e ajustado com dados gerados pelos Qwen2.5-7B/14B/32B-Instruct (todos Apache-2.0, verificado na API do HF). Não achei restrição herdada.
> - **Não verificado:** o texto do pré-treino (GigaVerbo-v2) mistura fontes com licenças diversas, algumas não comerciais (ex.: XL-Sum CC-BY-NC-SA, Bactrian-X CC-BY-NC). O modelo pode, em tese, regurgitar trechos; use as saídas como paráfrase/explicação nova, filtre cópias literais e, se o Apolo for comercial, peça parecer jurídico.
> - **Qualidade é o risco real**: o Tucano2-0.5B tem HumanEval 10% e o 1,5B 26% (pass@1). Explicações de código erradas **envenenam** o córtex. Por isso este caderno marca tudo como `revisar` e traz filtros; revise uma amostra à mão antes de misturar no corpus.

In [ ]:
!pip -q install -U transformers accelerate

In [ ]:
import json, os, re, csv, time, random, hashlib
import torch
from google.colab import drive
drive.mount('/content/drive')
PASTA = '/content/drive/MyDrive/apolo/tucano_professor'
os.makedirs(PASTA, exist_ok=True)

# ---- CONFIGURAÇÃO ----
# 3.7B em bf16 usa ~7,6 GB de GPU (cabe na T4 de 16 GB); 1.5B usa ~3 GB.
MODEL_ID = "Polygl0t/Tucano2-qwen-3.7B-Instruct"     # ou ...-1.5B-Instruct para ir mais rápido
ARQ_PERGUNTAS = PASTA + "/perguntas.json"            # entrada do modo paráfrase
ARQ_CODIGOS   = PASTA + "/codigos.json"              # entrada do modo explicar código
N_PARAFRASES  = 5                                    # quantas paráfrases tentar por pergunta
SEMENTE = 42
random.seed(SEMENTE); torch.manual_seed(SEMENTE)

## Formato dos arquivos de entrada
**perguntas.json** (qualquer um destes formatos):
```json
{"saudacao": ["Oi, tudo bem?", "Bom dia"], "criar_lista": ["Como criar uma lista em Python?"]}
```
ou `[{"intencao": "saudacao", "pergunta": "Oi, tudo bem?"}, ...]` ou uma lista simples de textos.

**codigos.json**: `[{"id": "py_001", "linguagem": "python", "codigo": "for i in range(3):\n    print(i)"}, ...]`

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8 else torch.float16  # T4 (7.5) não tem bf16 nativo
tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype, device_map="auto").eval()

def gerar(prompt, n=1, temperatura=0.7, max_novos=256, top_p=0.95):
    """Gera n respostas amostradas para o mesmo prompt (diversidade para paráfrases / melhor-de-N)."""
    texto = tok.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)
    ent = tok(texto, return_tensors="pt").to(model.device)
    cfg = GenerationConfig(do_sample=True, temperature=temperatura, top_p=top_p, top_k=50, repetition_penalty=1.15,
                           max_new_tokens=max_novos, num_return_sequences=n, pad_token_id=tok.eos_token_id)
    with torch.no_grad():
        out = model.generate(**ent, generation_config=cfg)
    return [tok.decode(o[ent.input_ids.shape[1]:], skip_special_tokens=True).strip() for o in out]

## Filtro de caracteres (mesmo conjunto do `scripts/montar_corpus.py`)
O córtex do Apolo é por caractere: texto com caractere fora do vocabulário vira ruído. Estas constantes são cópia de `PERMITIDOS`/`TROCAS`.

In [ ]:
PERMITIDOS = set(
    "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"
    "áàâãäéèêëíìîïóòôõöúùûüçñÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ"
    "0123456789 \n.,;:!?()-'\"%/$+=*&@#_"
)
TROCAS = str.maketrans({
    "\u201c": '"', "\u201d": '"', "\u00ab": '"', "\u00bb": '"', "\u201e": '"', "\u2018": "'", "\u2019": "'",
    "\u00b4": "'", "`": "'", "\u2013": "-", "\u2014": "-", "\u2212": "-", "\u2010": "-", "\u2011": "-",
    "\u00a0": " ", "\u2009": " ", "\u200a": " ", "\u202f": " ", "\u2026": "...",
    "\u00ba": "o", "\u00aa": "a", "\u00b0": "o", "\t": " ",
})
def limpar_linha(l):
    l = re.sub(r" {2,}", " ", l.translate(TROCAS)).strip()
    return None if (not l or any(c not in PERMITIDOS for c in l)) else l

def norm(s): return re.sub(r"[^a-z0-9 ]", "", s.lower()).strip()

## Modo 1: paráfrases de perguntas (detector de intenções)

In [ ]:
def carregar_perguntas(caminho):
    d = json.load(open(caminho, encoding="utf-8"))
    pares = []
    if isinstance(d, dict):
        for intencao, lista in d.items():
            pares += [(intencao, q) for q in lista]
    else:
        for i, x in enumerate(d):
            if isinstance(x, str): pares.append(("", x))
            else: pares.append((x.get("intencao", ""), x.get("pergunta") or x.get("texto") or x.get("frase")))
    return pares

PROMPT_PARAFRASE = (
    "Reescreva a frase abaixo de uma forma diferente, em português do Brasil, mantendo EXATAMENTE o mesmo significado e a mesma intenção. "
    "Responda apenas com a nova frase, sem aspas, sem explicações e sem numeração.\n\nFrase: {q}"
)

def parafrasear(pergunta, n=N_PARAFRASES):
    brutas = gerar(PROMPT_PARAFRASE.format(q=pergunta), n=n, temperatura=0.9, max_novos=60)
    vistos, boas = {norm(pergunta)}, []
    for b in brutas:
        b = limpar_linha(b.split("\n")[0].strip().strip('"').strip())
        if not b or not (4 <= len(b) <= 200): continue
        if norm(b) in vistos: continue                        # igual ao original ou repetida
        if b.lower().startswith(("claro", "aqui", "frase", "nova frase")): continue
        vistos.add(norm(b)); boas.append(b)
    return boas

def rodar_parafrases(arq_in, saida_base):
    pares = carregar_perguntas(arq_in)
    linhas = []
    t0 = time.time()
    for k, (intencao, q) in enumerate(pares, 1):
        for p in parafrasear(q):
            linhas.append({"intencao": intencao, "original": q, "parafrase": p, "modelo": MODEL_ID, "status": "revisar"})
        if k % 10 == 0: print(f"{k}/{len(pares)} perguntas, {len(linhas)} paráfrases, {time.time()-t0:.0f}s")
    with open(saida_base + ".json", "w", encoding="utf-8") as f: json.dump(linhas, f, ensure_ascii=False, indent=1)
    with open(saida_base + ".csv", "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["intencao", "original", "parafrase", "modelo", "status"]); w.writeheader(); w.writerows(linhas)
    print("gravado:", saida_base + ".json/.csv", "-", len(linhas), "paráfrases")
    return linhas

# Se ainda não existir entrada, cria um exemplo mínimo para testar:
if not os.path.exists(ARQ_PERGUNTAS):
    json.dump({"saudacao": ["Oi, tudo bem?", "Bom dia!"],
               "criar_lista_python": ["Como criar uma lista em Python?"],
               "join_sql": ["O que é um JOIN em SQL?"]},
              open(ARQ_PERGUNTAS, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
parafrases = rodar_parafrases(ARQ_PERGUNTAS, PASTA + "/parafrases")
for p in parafrases[:10]: print(p["intencao"], "|", p["original"], "->", p["parafrase"])

## Modo 2: explicar código em português
Gera uma explicação curta por bloco. `status: revisar` em tudo; a verificação automática só descarta o óbvio
(vazia, em outro idioma, copia o código, repete o prompt). Para melhor-de-N, aumente `N_EXPLICACOES` e escolha com o critério da célula seguinte
(o Tucano 3,7B como juiz é fraco para código: HumanEval 47,6%; prefira juiz heurístico ou revisão humana).

In [ ]:
N_EXPLICACOES = 1       # >1 gera candidatas; a célula escolhe a "melhor" por heurística simples

PROMPT_EXPLICAR = (
    "Explique em português do Brasil, de forma simples e curta (3 a 5 frases), o que o código {ling} abaixo faz. "
    "Não reescreva o código. Não use listas.\n\n```{ling}\n{codigo}\n```"
)

def pontuar(expl, codigo):
    """Heurística barata (não substitui revisão): tamanho razoável, português, sem copiar o código, termina em pontuação."""
    s = 0.0
    if 120 <= len(expl) <= 700: s += 1
    if re.search(r"\b(o|a|que|um|uma|de|para|com)\b", expl.lower()): s += 1
    if expl.rstrip()[-1:] in ".!?": s += 1
    linhas_cod = [l.strip() for l in codigo.splitlines() if len(l.strip()) > 8]
    if linhas_cod and any(l in expl for l in linhas_cod): s -= 1
    if "```" in expl: s -= 1
    return s

def explicar(item):
    ling = item.get("linguagem", "") or "de programação"
    cands = gerar(PROMPT_EXPLICAR.format(ling=ling, codigo=item["codigo"]), n=N_EXPLICACOES, temperatura=0.4 if N_EXPLICACOES == 1 else 0.8, max_novos=260)
    cands = [c for c in (limpar_linha(" ".join(c.split())) for c in cands) if c]   # uma linha, só caracteres permitidos
    if not cands: return None
    return max(cands, key=lambda c: pontuar(c, item["codigo"]))

def rodar_explicacoes(arq_in, saida_base):
    itens = json.load(open(arq_in, encoding="utf-8"))
    out = []
    for k, it in enumerate(itens, 1):
        e = explicar(it)
        out.append({"id": it.get("id", k), "linguagem": it.get("linguagem", ""), "codigo": it["codigo"],
                    "explicacao": e, "modelo": MODEL_ID, "status": "revisar" if e else "falhou"})
        if k % 10 == 0: print(f"{k}/{len(itens)}")
    with open(saida_base + ".json", "w", encoding="utf-8") as f: json.dump(out, f, ensure_ascii=False, indent=1)
    with open(saida_base + ".csv", "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["id", "linguagem", "codigo", "explicacao", "modelo", "status"]); w.writeheader(); w.writerows(out)
    print("gravado:", saida_base + ".json/.csv")
    return out

if not os.path.exists(ARQ_CODIGOS):
    json.dump([{"id": "py_001", "linguagem": "python", "codigo": "for i in range(3):\n    print(i * 2)"},
               {"id": "sql_001", "linguagem": "sql", "codigo": "SELECT nome, COUNT(*) FROM pedidos GROUP BY nome;"}],
              open(ARQ_CODIGOS, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
expl = rodar_explicacoes(ARQ_CODIGOS, PASTA + "/explicacoes_codigo")
for e in expl: print(e["id"], "->", e["explicacao"])

## Depois: como usar a saída
1. Abra `parafrases.csv` / `explicacoes_codigo.csv` e revise uma amostra (30 a 50 linhas) trocando `revisar` por `ok` ou `ruim`.
2. Só leve para o corpus o que estiver `ok` (ou o lote inteiro se a taxa de acerto da amostra for alta o bastante).
3. Formato de corpus do Apolo: um texto por linha, blocos separados por linha em branco; mantenha o peso baixo para não deixar o córtex de 85M decorar o estilo do Tucano.